# Aggregate TELL Load Data by NERC TPL-08 Region

This notebook takes an existing set of simulations using the Total ELectricity Loads (TELL) model and aggregates the BA-level data within each NERC TPL-08 region. The original TELL simulations were conducted using code stored here: https://github.com/cdburley/foresight_energy_drought_evolution_loads. We are using the 2024 runs which passes 45 years (1980-2024) of weather through the ML-based load models trained on data from 2021-2023. The loads are scaled to match the 2024 annual energy values from the IM3 rcp45hotter_ssp3 GCAM-USA run which tracks reasonably well with load growth over the last few years. All years have the same annual energy but different hour-to-hour variability.

The absolute values of the loads should be taken with a grain of salt. TELL BAs also do not directly align with the NERC TPL-08 regions. Notably, TELL models MISO and SPP as single BAs whereas they're split up in the TPL-08 shapefiles. For lack of more information I just took the MISO loads and split them in half to create hourly loads for MISO-N and MISO-S. Same for SPP. Annual peak loads by interconnection are in the ballpark of publicly available information on the peaks.


In [3]:
# Start by importing the packages we need:
import os
import glob
import datetime
import warnings

import pandas as pd
import numpy as np


## Suppress Future Warnings


In [4]:
warnings.simplefilter(action='ignore', category=FutureWarning)


## Set the Directory Structure

In [10]:
# Identify the data input and image output directory:
tell_load_data_dir =  '/Users/burl878/Documents/Code/code_repos/foresight_energy_drought_evolution_loads/data/tell_output/rcp45hotter_ssp3/2024/'
ba_mapping_data_dir = '/Users/burl878/Documents/Code/code_repos/gdo_climate_toolsuite_visualizations/data/'
data_output_dir = '/Users/burl878/Documents/Code/code_repos/gdo_climate_toolsuite_visualizations/data/load_data/'


## Create a Function to Aggregate TELL BA-Level Data for Each NERC TPL-08 Region


In [125]:
def aggregate_tell_load_data(start_year: int, end_year: int, tell_load_data_dir: str, ba_mapping_data_dir: str, data_output_dir: str):

    # Read in TELL BA to NERC TPL-08 region mapping:
    mapping = pd.read_csv((ba_mapping_data_dir + 'nerc_tpl08_region_to_TELL_BA_mapping.csv'))
    
    # Extract the list of NERC TPL-08 region names:
    nerc_tpl08_regions = mapping['short_name'].unique()
                
    # Loop over the years of TELL load data:
    for year in range(start_year, end_year, 1):
        # Read in the raw TELL BA output file for the TELL weather year:
        ba_df = pd.read_csv((tell_load_data_dir + 'TELL_Balancing_Authority_Hourly_Load_Data_' + str(year) + '_Scaled_2024.csv'))

        # Loop over each of the NERC TPL-08 regions:
        for i in range(len(nerc_tpl08_regions)):
        
            # Make a list of the BAs that map to that NERC TPL-08 region:
            tell_subset = mapping[(mapping['short_name'] == nerc_tpl08_regions[i])].copy()
            tell_bas = tell_subset['tell_ba'].unique()
            
            # Subset the TELL output file to just the BAs in that NERC TPL-08 region:
            ba_subset_df = ba_df[ba_df['BA_Code'].isin(tell_bas)].copy()

            # Sum the TELL BA-level loads across all BAs in that NERC TPL-08 region:
            ba_subset_df['Region_Load_MWh'] = ba_subset_df.groupby('Time_UTC')['Scaled_TELL_BA_Load_MWh'].transform('sum').round(2)

            # Only keep the variable we need, drop duplicates, and sort chronologically:
            region_df = ba_subset_df[['Time_UTC','Region_Load_MWh']]
            region_df = region_df.drop_duplicates().sort_values('Time_UTC')
            region_df.reset_index(inplace=True, drop=True)
            
            # Rename the regional load to the region name:
            region_df.rename(columns={'Region_Load_MWh': nerc_tpl08_regions[i]}, inplace=True)
        
            # Aggregate the output into a new dataframe:
            if i == 0:
               year_df = region_df
            else:
               year_df = year_df.merge(region_df, on=['Time_UTC'])

            # Clean up and move to the next region:
            del tell_subset, tell_bas, ba_subset_df, region_df
        
        # Aggregate the output into a new dataframe:
        if year == start_year:
           output_df = year_df
        else:
           output_df = pd.concat([output_df, year_df])

        # Clean up and move to the next year:
        del ba_df, i, year_df

    # Sum up the regions in each interconnection:
    output_df['WECC'] = (output_df['CA'] + output_df['PNW'] + output_df['GB'] + output_df['SW'] + output_df['RM']).round(2)
    output_df['EIC'] = (output_df['SPP'] + output_df['MISO'] + output_df['SERC'] + output_df['FL'] + output_df['PJM'] + output_df['NYISO'] + output_df['ISONE']).round(2)

    # Split MISO and SPP into north and south components using a 50/50 split:
    output_df['MISO-N'] = (output_df['MISO'] * 0.5).round(2)
    output_df['MISO-S'] = (output_df['MISO'] * 0.5).round(2)
    output_df['SPP-N'] = (output_df['SPP'] * 0.5).round(2)
    output_df['SPP-S'] = (output_df['SPP'] * 0.5).round(2)
    
    # Only keep the columns we need:
    output_df = output_df[['Time_UTC', 'EIC', 'ERCOT', 'WECC', 'CA', 'FL', 'GB', 'ISONE', 'MISO-N', 'MISO-S', 'NYISO', 'PJM', 'PNW', 'RM', 'SERC', 'SPP-N', 'SPP-S', 'SW']].copy()
    
    # Reset the index value:
    output_df.reset_index(inplace=True, drop=True)
    
    # Set the output filename:
    output_filename = ('NERC_Region_Hourly_Loads_' + str(start_year) + '_to_' + str(end_year) + '.csv')
        
    # Write out the dataframe to a .csv file:
    output_df.to_csv((os.path.join(data_output_dir, output_filename)), sep=',', index=False)
    
    # Return the output dataframe:
    return output_df


In [126]:
# Execute the function:
output_df = aggregate_tell_load_data(start_year = 1980,
                                     end_year = 2025, 
                                     tell_load_data_dir = tell_load_data_dir,
                                     ba_mapping_data_dir = ba_mapping_data_dir,
                                     data_output_dir = data_output_dir)

output_df


,Time_UTC,EIC,ERCOT,WECC,CA,FL,GB,ISONE,MISO-N,MISO-S,NYISO,PJM,PNW,RM,SERC,SPP-N,SPP-S,SW
0,1980-01-01 00:00:00,353874.11,32623.51,87132.15,32247.82,26457.89,14862.78,19788.41,37220.21,37220.21,20591.75,102809.05,23161.78,5727.97,72910.07,18438.26,18438.26,11131.80
1,1980-01-01 01:00:00,351746.57,33786.16,90681.69,34213.55,25571.52,15156.99,18909.66,37512.24,37512.24,20054.93,101293.93,23862.98,5798.68,73720.46,18585.79,18585.79,11649.49
2,1980-01-01 02:00:00,347782.68,34962.04,92534.55,35216.13,24711.22,15344.91,18054.70,37313.76,37313.76,19487.78,99459.65,24035.02,5896.30,74145.65,18648.08,18648.08,12042.19
3,1980-01-01 03:00:00,338810.55,34769.00,93778.17,36314.99,23628.76,15437.77,17267.82,36396.25,36396.25,18923.37,97392.05,24088.93,5795.51,71997.69,18404.18,18404.18,12140.97
4,1980-01-01 04:00:00,330037.08,34252.57,92776.70,36307.27,22611.40,15198.75,16467.52,35440.06,35440.06,18308.29,95599.34,23899.59,5549.62,69727.79,18221.31,18221.31,11821.47
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
394481,2024-12-31 17:00:00,349837.01,36925.27,97818.56,36985.72,28373.71,15328.14,15746.53,38600.07,38600.07,19409.74,99916.80,28726.88,5625.13,67185.16,21002.46,21002.46,11152.69
394482,2024-12-31 18:00:00,350277.97,36440.13,96138.13,36512.83,29707.69,15248.05,15775.15,38631.92,38631.92,19286.31,99620.06,28067.73,5591.78,66854.53,20885.20,20885.20,10717.74
394483,2024-12-31 19:00:00,351628.91,35959.63,94567.08,35950.77,30862.02,15150.44,16180.02,38575.44,38575.44,19215.51,100244.41,27437.31,5579.99,66573.41,20701.33,20701.33,10448.57
394484,2024-12-31 20:00:00,353273.58,35860.09,93466.14,35490.03,31428.13,15084.50,16831.05,38573.26,38573.26,19276.06,100944.22,26945.69,5588.97,66693.47,20477.06,20477.06,10356.95


## Create a Function to Calculate the Peak Load by Day


In [127]:
def process_daily_data(start_year: int, end_year: int, data_output_dir: str):

    # Read in data processed using the function created above:
    load_df = pd.read_csv((data_output_dir + 'NERC_Region_Hourly_Loads_' + str(start_year) + '_to_' + str(end_year) + '.csv'))

    # Set the time as datetime variable:
    load_df['Time_UTC'] = pd.to_datetime(load_df['Time_UTC'])

    # Extract the day from the datetime variable:
    load_df['Date'] = load_df['Time_UTC'].dt.date

    # Compute the daily max for each region:
    load_df['EIC_Max'] = load_df.groupby('Date')['EIC'].transform('max').round(2)
    load_df['ERCOT_Max'] = load_df.groupby('Date')['ERCOT'].transform('max').round(2)
    load_df['WECC_Max'] = load_df.groupby('Date')['WECC'].transform('max').round(2)
    load_df['CA_Max'] = load_df.groupby('Date')['CA'].transform('max').round(2)
    load_df['FL_Max'] = load_df.groupby('Date')['FL'].transform('max').round(2)
    load_df['GB_Max'] = load_df.groupby('Date')['GB'].transform('max').round(2)
    load_df['ISONE_Max'] = load_df.groupby('Date')['ISONE'].transform('max').round(2)
    load_df['MISO-N_Max'] = load_df.groupby('Date')['MISO-N'].transform('max').round(2)
    load_df['MISO-S_Max'] = load_df.groupby('Date')['MISO-S'].transform('max').round(2)
    load_df['NYISO_Max'] = load_df.groupby('Date')['NYISO'].transform('max').round(2)
    load_df['PJM_Max'] = load_df.groupby('Date')['PJM'].transform('max').round(2)
    load_df['PNW_Max'] = load_df.groupby('Date')['PNW'].transform('max').round(2)
    load_df['RM_Max'] = load_df.groupby('Date')['RM'].transform('max').round(2)
    load_df['SERC_Max'] = load_df.groupby('Date')['SERC'].transform('max').round(2)
    load_df['SPP-N_Max'] = load_df.groupby('Date')['SPP-N'].transform('max').round(2)
    load_df['SPP-S_Max'] = load_df.groupby('Date')['SPP-S'].transform('max').round(2)
    load_df['SW_Max'] = load_df.groupby('Date')['SW'].transform('max').round(2)
    
    # Subset and reorder the columns, drop duplicates, and sort chronologically:
    output_df = load_df[['Date', 'EIC_Max', 'ERCOT_Max', 'WECC_Max', 'CA_Max', 'FL_Max', 'GB_Max', 'ISONE_Max', 'MISO-N_Max', 'MISO-S_Max', 'NYISO_Max',
                         'PJM_Max', 'PNW_Max', 'RM_Max', 'SERC_Max', 'SPP-N_Max', 'SPP-S_Max', 'SW_Max']]
    output_df = output_df.drop_duplicates()
    output_df = output_df.sort_values('Date')

    # Rename the columns for simplicity:
    output_df.rename(columns={'WECC_Max': 'WECC', 'EIC_Max': 'EIC', 'ERCOT_Max': 'ERCOT', 'CA_Max': 'CA', 'FL_Max': 'FL', 'GB_Max': 'GB', 'ISONE_Max': 'ISONE',
                              'MISO-N_Max': 'MISO-N', 'MISO-S_Max': 'MISO-S', 'NYISO_Max': 'NYISO', 'PJM_Max': 'PJM', 'PNW_Max': 'PNW', 'RM_Max': 'RM', 'SERC_Max': 'SERC',
                              'SPP-N_Max': 'SPP-N', 'SPP-S_Max': 'SPP-S', 'SW_Max': 'SW'}, inplace=True)

    # Replace zeros with NaN and reset the index value:
    output_df.replace(0, np.nan, inplace=True)
    output_df.reset_index(inplace=True, drop=True)
    
    # Set the output filename:
    output_filename = ('NERC_Region_Daily_Peak_' + str(start_year) + '_to_' + str(end_year) + '.csv')
        
    # Write out the dataframe to a .csv file:
    output_df.to_csv((os.path.join(data_output_dir, output_filename)), sep=',', index=False)
    
    return output_df


In [128]:
# Execute the function:
output_df = process_daily_data(start_year = 1980,
                               end_year = 2025, 
                               data_output_dir = data_output_dir)

output_df


,Date,EIC,ERCOT,WECC,CA,FL,GB,ISONE,MISO-N,MISO-S,NYISO,PJM,PNW,RM,SERC,SPP-N,SPP-S,SW
0,1980-01-01,362619.43,39365.68,93778.17,36314.99,27141.72,15437.77,19788.41,37817.61,37817.61,20591.75,107076.32,24108.32,5896.30,82083.66,19356.48,19356.48,12140.97
1,1980-01-02,396263.88,38581.67,99296.03,40266.37,33626.28,15599.24,20369.61,40808.84,40808.84,22260.08,113198.63,26575.12,6020.43,92794.43,20009.23,20009.23,13071.11
2,1980-01-03,396694.62,37237.92,100920.35,40336.83,32446.83,15842.23,20424.75,41242.68,41242.68,22780.07,114211.06,27927.18,6256.27,89723.07,20850.45,20850.45,13647.37
3,1980-01-04,392752.49,42735.66,102632.08,40286.48,29177.85,15738.40,21414.44,42337.90,42337.90,23640.68,117627.57,28715.61,6239.50,82156.89,21055.26,21055.26,13636.86
4,1980-01-05,377758.46,38827.60,99023.36,39062.62,27363.73,15171.27,20916.59,40227.07,40227.07,22736.75,111352.39,26557.93,5968.13,86627.69,19341.20,19341.20,12364.64
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16432,2024-12-27,377057.04,37177.33,98358.39,39620.80,30152.86,15216.03,21223.86,38052.12,38052.12,23503.29,112887.87,25768.33,5704.65,77057.67,18943.30,18943.30,12607.90
16433,2024-12-28,349917.08,36333.10,94359.03,36977.27,28915.70,15106.88,19375.39,36023.86,36023.86,21645.64,102564.48,25175.34,5478.65,70600.56,18099.38,18099.38,12042.87
16434,2024-12-29,335803.82,36723.75,93734.13,37115.11,28415.59,14921.13,18281.30,35840.21,35840.21,20249.56,96164.15,24574.60,5454.86,65778.19,17859.05,17859.05,11950.41
16435,2024-12-30,353145.38,37449.37,97836.89,39274.56,30792.10,14922.36,18722.99,38552.00,38552.00,21265.36,102480.15,27015.04,5662.35,69790.98,19251.42,19251.42,12299.84
